<a href="https://colab.research.google.com/github/Jalilnkh/PyTorch-with-Examples-2024/blob/parts/speech_to_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import re
import librosa
import torch
import pandas as pd
import torchaudio
import numpy as np
from torch.utils.data import DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MAX_AUDIO_LENGTH = 160000  # Maximum audio length in samples (can be adjusted)

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Directory Paths
MAIN_DIR = '/content/drive/MyDrive/KartalOl Corpus/Dataset/Speech Recognition dataset/Work with Farhan/dataset'




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
os.listdir(MAIN_DIR)

['sentences', 'voices']

In [14]:
def AZBDataset(dataset):
    """
    Custom function to format AZB speech dataset for DataLoader.
    """
    def get_item(idx):
        sample = dataset[idx]
        return {
            'input_values': sample['audio_array'],
            'labels': sample['normalized_text']
        }

    return [get_item(i) for i in range(len(dataset))]

def pad_or_truncate(audio_array, max_length=MAX_AUDIO_LENGTH):
    """Pad or truncate an audio array to a fixed length."""
    if len(audio_array) > max_length:
        return audio_array[:max_length]
    else:
        return np.pad(audio_array, (0, max_length - len(audio_array)), 'constant')

def get_person_folders(main_dir):
    """Retrieve all person folders ending with '-bot'."""
    return [
        os.path.join(main_dir, folder)
        for folder in os.listdir(main_dir)
        if os.path.isdir(os.path.join(main_dir, folder)) and folder.endswith('-bot')
    ]

def load_csv_files(name_csv_dir):
    """Load CSV files into a dictionary with PersonID as keys."""
    csv_dict = {}
    for file in os.listdir(name_csv_dir):
        if file.endswith('.csv'):
            person_id = os.path.splitext(file)[0]
            csv_path = os.path.join(name_csv_dir, file)
            try:
                df = pd.read_csv(csv_path)
                csv_dict[person_id] = df
            except Exception as e:
                print(f"Error reading {csv_path}: {e}")
    return csv_dict

def normalize_text(text):
    """Normalize text by lowercasing and removing punctuation."""
    text = text.lower()
    text = text.translate(str.maketrans('', '', '"\'""!.,?;:'))  # Remove basic punctuation
    text = ' '.join(text.split())  # Remove extra whitespace
    return text

def create_tts_dataset(main_dir, sampling_rate=16000):
    dataset = []
    person_folders = get_person_folders(main_dir+'/voices')
    name_csv_dir = os.path.join(main_dir, 'sentences')
    csv_dict = load_csv_files(name_csv_dir)

    for person_folder in person_folders:
        folder_name = os.path.basename(person_folder)
        person_id = folder_name.replace('-bot', '')

        if person_id not in csv_dict:
            print(f"CSV for PersonID {person_id} not found. Skipping this person.")
            continue

        df = csv_dict[person_id]

        for file in os.listdir(person_folder):
            if file.endswith('.wav'):
                try:
                    parts = os.path.splitext(file)[0].split('_')
                    if len(parts) != 4:
                        continue
                    file_person_id, sentence_id, number, book_id = parts
                    match = re.search(r'ID(\d+)', sentence_id)
                    if match:
                        sentence_id = int(match.group(1))
                    if 'f' in file_person_id:
                        gender = 'female'
                    else:
                        gender = 'male'

                except Exception as e:
                    continue

                if file_person_id != person_id:
                    continue

                matching_rows = df.iloc[sentence_id]
                if matching_rows.empty:
                    continue

                text = matching_rows['Sentence']
                normalized = normalize_text(text)

                audio_path = os.path.join(person_folder, file)

                try:
                    audio_array, sr = librosa.load(audio_path, sr=sampling_rate)
                    audio_array = pad_or_truncate(audio_array)
                except Exception as e:
                    continue

                dataset.append({
                    'person_id': person_id,
                    'gender': gender,
                    'sentence_id': sentence_id,
                    'number_in_book': number,
                    'book_id': book_id,
                    'file': file,
                    'audio_path': audio_path,
                    'audio_array': audio_array,
                    'sampling_rate': sr,
                    'text': text,
                    'normalized_text': normalized
                })

    return dataset

def train_model():
    # Load Whisper Processor and Model
    processor = WhisperProcessor.from_pretrained("openai/whisper-large-v2")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v2")
    model.config.forced_decoder_ids = None  # No forced language ID

    # Create dataset and DataLoader
    print("Creating dataset...")
    dataset = create_tts_dataset(MAIN_DIR)
    azb_dataset = AZBDataset(dataset)
    train_loader = DataLoader(azb_dataset, batch_size=4, shuffle=True)

    # Test if data is loading correctly
    batch = next(iter(train_loader))
    print("Sample batch from train loader:", batch)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    model.train()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    for epoch in range(5):  # 5 epochs for simplicity
        total_loss = 0
        for batch in train_loader:
            inputs = processor(batch['input_values'], return_tensors="pt", sampling_rate=16000, padding=True)
            labels = processor(text=batch['labels'], return_tensors="pt", padding=True).input_ids

            inputs = inputs['input_values'].to(device)
            labels = labels.to(device)

            outputs = model(input_features=inputs, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1} / 5 - Loss: {total_loss:.4f}")
       # Save the trained model
    save_path = '/content/drive/MyDrive/whisper_model'
    model.save_pretrained(save_path)
    processor.save_pretrained(save_path)
    print(f"Model saved to {save_path}")

def transcribe_audio(audio_path):
    processor = WhisperProcessor.from_pretrained("openai/whisper-large-v2")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v2")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    audio_array, sr = librosa.load(audio_path, sr=16000)
    inputs = processor(audio_array, return_tensors="pt", sampling_rate=16000, padding=True)
    input_values = inputs['input_values'].to(device)

    with torch.no_grad():
        predicted_ids = model.generate(input_values)

    transcription = processor.decode(predicted_ids[0], skip_special_tokens=True)
    return transcription



In [ ]:
train_model()


PYDEV DEBUGGER WARNING:
sys.settrace() should not be used when the debugger is being used.
This may cause the debugger to stop working correctly.
If this is needed, please check: 
http://pydev.blogspot.com/2007/06/why-cant-pydev-debugger-work-with.html
to see how to restore the debug tracing back correctly.
Call Location:
  File "/usr/lib/python3.10/bdb.py", line 336, in set_trace
    sys.settrace(self.trace_dispatch)



Creating dataset...
> <ipython-input-14-1281c3b418a7>(66)create_tts_dataset()
     64         df = csv_dict[person_id]
     65 
---> 66         for file in os.listdir(person_folder):
     67             if file.endswith('.wav'):
     68                 try:




PYDEV DEBUGGER WARNING:
sys.settrace() should not be used when the debugger is being used.
This may cause the debugger to stop working correctly.
If this is needed, please check: 
http://pydev.blogspot.com/2007/06/why-cant-pydev-debugger-work-with.html
to see how to restore the debug tracing back correctly.
Call Location:
  File "/usr/lib/python3.10/bdb.py", line 361, in set_quit
    sys.settrace(None)


PYDEV DEBUGGER WARNING:
sys.settrace() should not be used when the debugger is being used.
This may cause the debugger to stop working correctly.
If this is needed, please check: 
http://pydev.blogspot.com/2007/06/why-cant-pydev-debugger-work-with.html
to see how to restore the debug tracing back correctly.
Call Location:
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/debugger.py", line 1075, in cmdloop
    sys.settrace(None)



--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user
> <ipython-input-14-1281c3b418a7>(66)create_tts_dataset()
     64         df = csv_dict[person_id]
     65 
---> 66         for file in os.listdir(person_folder):
     67             if file.endswith('.wav'):
     68                 try:

--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user
> <ipython-input-14-1281c3b418a7>(66)create_tts_dataset()
     64         df = csv_dict[person_id]
     65 
---> 66         for file in os.listdir(person_folder):
     67             if file.endswith('.wav'):
     68                 try:

--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user
> <ipython-input-14-1281c3b418a7>(66)create_tts_dataset()
     64         df = csv_dict[person_id]
     65 
---> 66         for file in os.listdir(person_folder):
     67             if file.endswith('.wav'):
     68                 try:

--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user
> <ipython-input-14-1281c3b418a7>(6

In [ ]:
# Example Transcription
sample_audio_path = '/path/to/your/audio/file.wav'
transcription = transcribe_audio(sample_audio_path)
print("Transcription:", transcription)